# Exploratory Data Analysis (EDA)

Inspect the shape of our dataset

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train_df = pd.read_csv('data/UNSW_NB15_training-set.csv')
test_df = pd.read_csv('data/UNSW_NB15_testing-set.csv')

train_df.drop('id', axis=1, inplace=True, errors='ignore')
test_df.drop('id', axis=1, inplace=True, errors='ignore')

print(f"Training set shape: {train_df.shape}")
print(f"Testing set shape: {test_df.shape}")

print("\n--- Training Set Info ---")
train_df.info()

print("\n--- Attack Category Distribution ---")
print(train_df['attack_cat'].value_counts())

print("\n--- Binary Target Distribution ---")
print(train_df['label'].value_counts())

Training set shape: (175341, 44)
Testing set shape: (82332, 44)

--- Training Set Info ---
<class 'pandas.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 44 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   dur                175341 non-null  float64
 1   proto              175341 non-null  str    
 2   service            175341 non-null  str    
 3   state              175341 non-null  str    
 4   spkts              175341 non-null  int64  
 5   dpkts              175341 non-null  int64  
 6   sbytes             175341 non-null  int64  
 7   dbytes             175341 non-null  int64  
 8   rate               175341 non-null  float64
 9   sttl               175341 non-null  int64  
 10  dttl               175341 non-null  int64  
 11  sload              175341 non-null  float64
 12  dload              175341 non-null  float64
 13  sloss              175341 non-null  int64  
 14  dloss              1

Based on the type of data, we'll use sample weight bias to address the class imbalance

In [3]:
import numpy as np
from sklearn.utils.class_weight import compute_sample_weight

# The UNSW dataset often has trailing spaces in the attack_cat column
train_df['attack_cat'] = train_df['attack_cat'].str.strip()
test_df['attack_cat'] = test_df['attack_cat'].str.strip()

# Compute sample weights for the multi-class target to penalize majority class bias
sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=train_df['attack_cat']
)

print(f"Sample weights shape: {sample_weights.shape}")
print(f"Unique weight values: {np.unique(sample_weights)}")

Sample weights shape: (175341,)
Unique weight values: [  0.31310893   0.4383525    0.5250831    0.96425979   1.42972114
   1.67134687   8.76705     10.04243986  15.47581642 134.87769231]


## Categorical Encoding

one hot encoding, with all unique protocols grouped together

In [4]:
cat_cols = ['proto', 'service', 'state']

# Keep the top 10 protocols, label the rest as 'other' to prevent dimensionality explosion
top_protos = train_df['proto'].value_counts().nlargest(10).index
train_df['proto'] = train_df['proto'].where(train_df['proto'].isin(top_protos), 'other')
test_df['proto'] = test_df['proto'].where(test_df['proto'].isin(top_protos), 'other')

# Apply one-hot encoding
train_encoded = pd.get_dummies(train_df, columns=cat_cols)
test_encoded = pd.get_dummies(test_df, columns=cat_cols)

# Align columns to ensure the test set matches the training set features perfectly
train_encoded, test_encoded = train_encoded.align(test_encoded, join='inner', axis=1)

print(f"Encoded training set shape: {train_encoded.shape}")
print(f"Encoded testing set shape: {test_encoded.shape}")

Encoded training set shape: (175341, 70)
Encoded testing set shape: (82332, 70)


## Normalization
Probably optional for XGBoost, but might be needed if we decide to try KNN or other models

In [5]:
from sklearn.preprocessing import StandardScaler

# Separate targets from features
targets = ['label', 'attack_cat']

X_train = train_encoded.drop(columns=targets)
y_train_multi = train_encoded['attack_cat']
y_train_binary = train_encoded['label']

X_test = test_encoded.drop(columns=targets)
y_test_multi = test_encoded['attack_cat']
y_test_binary = test_encoded['label']

# Identify continuous columns (excluding the one-hot encoded boolean/uint8 columns)
continuous_cols = [col for col in X_train.columns if not any(col.startswith(prefix) for prefix in ['proto_', 'service_', 'state_'])]

# Initialize the scaler
scaler = StandardScaler()

# Fit ONLY on the training data to prevent data leakage, then transform both
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("Scaling complete.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Scaling complete.
X_train shape: (175341, 68)
X_test shape: (82332, 68)


# XGBoost Training

In [ ]:
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import classification_report

# XGBoost strictly requires numerical labels starting from 0
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train_multi)
y_test_enc = le.transform(y_test_multi)

# Initialize the classifier
xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1 
)

print("Training XGBoost model... this may take a minute.")

# Fit the model using the sample weights to handle the class imbalance
xgb_model.fit(
    X_train, 
    y_train_enc, 
    sample_weight=sample_weights
)

# Generate predictions
y_pred_enc = xgb_model.predict(X_test)

# Convert integer predictions back to original string labels for readable metrics
y_pred = le.inverse_transform(y_pred_enc)

print("\n--- Classification Report ---")
print(classification_report(y_test_multi, y_pred, digits=4))